# Machine Learning Optimization

### **All exercises will be using your chosen dataset**

## Step 1: Import your dataset using the tutorial from the slides

## Step 2: Install necessary libraries. We will be using scikit-learn again for week 4

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.svm import SVR, SVC
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.calibration import CalibratedClassifierCV, calibration_curve, CalibrationDisplay
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score, accuracy_score, classification_report, ConfusionMatrixDisplay

# Using scikit learn's diabetes dataset for the solutions
from sklearn.datasets import load_diabetes

# Load the diabetes dataset
diabetes_sklearn = load_diabetes()

# Convert the dataset to a DataFrame
df = pd.DataFrame(data=diabetes_sklearn.data,
                           columns=diabetes_sklearn.feature_names)

# Add target variable to the DataFrame
df['target'] = diabetes_sklearn.target

## Exercises

### Scikit-Learn Docs, ML Interpretability Book, and Demo for Reference

#### [Scikit-Learn](https://scikit-learn.org/stable/)
#### [ML Interpretability](https://christophm.github.io/interpretable-ml-book/)
#### [Demo](https://mdst-ai-in-healthcare.streamlit.app/)


Import your **preprocessed dataset** from week 3

In [ ]:
# TODO: Load your dataset with pandas

# df = pd.read_csv('name_of_dataset.csv')

### Exercise 1: Cross Validation Evaluations

Rather than evaluating a model on a simple train-test split like we did in week 3, we can use cross validation to provide a more robust and accurate estimate of model performance. This method splits the data up into k subsets (k can be tuned) and is trained k times with k-1 subsets used for training and 1 subset for testing.

In [ ]:
# TODO: Define the features and the target data
X = df.drop(columns=["target"])
y = df["target"]

In [ ]:
# TODO: initialize the model that you used in week 3. Either LinearRegression() or LogisticRegression()
cv_model = LinearRegression()

Now we will calculate the cross validation scores using cross_val_score(). This outputs an array of k scores. These scores depict how well the model generalizes to new, unseen data. The defualt scoring for **regression** is **R^2** and the default for **classification** is **accuracy**.

In [ ]:
# TODO: Calculate the cross validation score for this model on your data (use k=5)
scores = cross_val_score(cv_model, X, y, cv=5)

In [ ]:
# TODO: Print the array of scores as well as the mean and standard deviation using np.mean() and np.std()
print(scores)
print(np.mean(scores))
print(np.std(scores))

[0.42955615 0.52259939 0.48268054 0.42649776 0.55024834]
0.48231643590864215
0.04926857751190387


How well does the model that you used generalize to unseen data? What factors might influence a model to not generalize well?
**The model doesn't generalize the best. It has an average of 50% R^2 score for all 5 cv's. A reason for poor generalization could be highly correlated features.**

### Exercise 2: Regularization

**Regularization** is a technique that helps prevent a model from overfitting. Remember that overfitting is when a model learns noise in the training data and doesn't generalize to patterns. Underfitting on the other hand is when a model is too simple and doesn't understand the underlying structure in the data. Regularization adds a penalty term to a models cost function. A cost function calculates the models average errors between the predictions and the actual values. This penalty constrains the model from learning noise in the data and instead learns the patterns and encourages the model to generalize to unseen data.

**Lasso Regression (L1 Regularizer):** Adds the absolute value of the coefficients to the cost function. Good for selecting quality features and simplifying model coefficients

**Ridge Regression (L2 Regularizer):** Adds the squared magnitude of all the coefficients to the cost function. This is best used to deal with multicollinearity issues.

**ElasticNet Regression:** Combines both L1 and L2 regression to the cost function. Gives the best of both worlds.

If you used **linear regression:**

We will be using the regularizer **ElasticNet Regression**

In [ ]:
# TODO: Initialize the ElasticNet model and use alpha=1 to start (ElasticNet())
# Higher values of alpha result in better generalization but can cause underfitting and lower values can lead to overfitting
elastic_net = ElasticNet(alpha=0.001)

In [ ]:
# TODO: using the model, run another cross validation evaluation
scores = cross_val_score(elastic_net, X, y, cv=5)

In [ ]:
# TODO: Print the array of scores as well as the mean and standard deviation using np.mean() and np.std()
print(scores)
print(np.mean(scores))
print(np.std(scores))

[0.40167721 0.51506351 0.48932433 0.44682643 0.52987169]
0.47655263368847445
0.04685958103934803


Has the generalization of your model improved?

If you used **logistic regression:**

The Logistic Regression Model automatically uses L2 regularization by default. You will be comparing the difference between the cross validation score on the L2 regularizer and the ElasticNet regularizer.

In [ ]:
# TODO: Initialize an L1 regularized logistic regression model using penalty="elasticnet"

# elastic_net = LogisticRegression(penalty="elasticnet")

In [ ]:
# TODO: using the model, run another cross validation evaluation

# scores = cross_val_score(elastic_net, X, y, cv=5)

In [ ]:
# TODO: Print the array of scores as well as the mean and standard deviation using np.mean() and np.std()

# print(scores)
# print(np.mean(scores))
# print(np.std(scores))


Has the generalization of your model improved?

Experiment with L1 and L2 Regularization and see which one gives the best cross validation scores.

### Exercise 3: Probability Calibration

Using predict_proba, we can calculate the probabilities of each class in classification. This can provide information in the confidence of a model for a particular class. However these probabilities don't reflect the empirical (real-world) probabilities of the classes that we are predicting. That is where probability calibration comes in. Probability calibration takes uncalibrated probabilities and maps them to more interpretable and accurate probabilities. For example, events with a predicted probability of 70% occur 70% of the time. The CalibratedClassifierCV in scikit-learn uses cross-validation to fit the model for optimizing the probabilities.

For the purposes of this exercise, we will be using a base classifier that has not been trained yet (CalibratedClassifierCV takes care of the training automatically)

In [ ]:
# TODO: Define a LogisticRegression() Model optionally using the a custom regularizer

logistic_model = LogisticRegression()

There are different methods of mapping uncalibrated to calibrated probabilities.

**Sigmoid (Platt Scaling):** Fits a Logistic Regression Model; ideal for smaller datasets

**Isotonic Regression:** A more flexible approach that works better on larger (>1000) datasets

In [ ]:
# TODO: Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# TODO: Create a CalibratedClassifierCV() object with cv=5 and estimator=logistic_model. Choose your desired method

# calibrated = CalibratedClassifierCV(estimator=logistic_model, method="isotonic", cv=5)

In [ ]:
# TODO: Fit the calibrated model using the train data

# calibrated.fit(X_train, y_train)

In [ ]:
# TODO: Next, fit the base logistic_model without any calibration

# logistic_model.fit(X_train, y_train)

In [ ]:
# TODO: Run this code block to graph the calibrated and uncalibrated probabilities for the model

# fig, ax = plt.subplots(figsize=(6, 6))

# # Default calibration curve (base model)
# CalibrationDisplay.from_estimator(
#     logistic_model,
#     X_test, y_test,
#     n_bins=10,
#     strategy="uniform",
#     name="uncalibrated",
#     ax=ax
# )

# # Calibrated curve (calibrated model)
# CalibrationDisplay.from_estimator(
#     model_cal,
#     X_test, y_test,
#     n_bins=10,
#     strategy="uniform",
#     name="calibrated",
#     ax=ax
# )

# ax.set_title("Calibration Plot")
# ax.set_xlabel("Predicted probability")
# ax.set_ylabel("Empirical Probability")
# ax.legend(loc="best")
# plt.show()

Did the probability calibration improve your models predicted probabilities? How well calibrated are your probabilities for the new model?

### Exercise 5: More Advanced Algorithms

So far, we have only been using logistic and linear regression which are very simple models that might not capture the underlying structure of the data you are working with. We will now be looking into other models the scikit-learn has to offer.

If your task is **regression**, ceate these models.

#### Random Forest Regressor:
Ensemble method that constucts many decision trees and averages the predictions for each tree

In [ ]:
# TODO: Initialize the model (RandomForestRegressor())
rf = RandomForestRegressor()

In [ ]:
# TODO: Calculate the cross validation score for this model on your data
scores = cross_val_score(rf, X, y, cv=5)

In [ ]:
# TODO: Print the array of scores as well as the mean and standard deviation using np.mean() and np.std()
print(scores)
print(np.mean(scores))
print(np.std(scores))

[0.37852832 0.50202659 0.42664264 0.37428243 0.41733057]
0.4197621091841578
0.046020268014330415


#### Support Vector Regressor
Similar to Support Vector Machine, but for regression. Finds the optimal hyperplane to the given data with a defined tolerance or margin for error rather than minimizing the error for every single data point.

In [ ]:
# TODO: Initialize the model (SVR())
svr = SVR()

In [ ]:
# TODO: Calculate the cross validation score for this model on your data
scores = cross_val_score(svr, X, y, cv=5)

In [ ]:
# TODO: Print the array of scores as well as the mean and standard deviation using np.mean() and np.std()
print(scores)
print(np.mean(scores))
print(np.std(scores))

[0.14739157 0.12560632 0.18203832 0.12242227 0.15658497]
0.14680869160894452
0.02182329158852706


If your task is **classification**, ceate these models.

#### Random Forest Classifier
Similar to the RandomForestRegressor, but instead of taking th average of the decision trees, it takes the most likely class given the class outputs of the individual trees.

In [ ]:
# TODO: Initialize the model (RandomForestClassifier())

# rf_class = RandomForestClassifier()

In [ ]:
# TODO: Calculate the cross validation score for this model on your data

# scores = cross_val_score(rf_class, X, y, cv=5)

In [ ]:
# TODO: Print the array of scores as well as the mean and standard deviation using np.mean() and np.std()

# print(scores)
# print(np.mean(scores))
# print(np.std(scores))

#### Support Vector Machine:
Finds an optimal hyperplane that maximizes the distance between the classes that the model is predicting.

In [ ]:
# TODO: Initialize the model (SVC())

# svc = SVC()

In [ ]:
# TODO: Calculate the cross validation score for this model on your data

# scores = cross_val_score(svc, X, y, cv=5)

In [ ]:
# TODO: Print the array of scores as well as the mean and standard deviation using np.mean() and np.std()

# print(scores)
# print(np.mean(scores))
# print(np.std(scores))

Which model had the best overall cross-validation score?

### Exercise 6: Further Exploration

Keep experimenting with different models, regularizers, etc...